In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. LOAD DATA
# =============================================
state_data = pd.read_csv("../data/state_dataset.csv", index_col="Date", parse_dates=True)
market_data = pd.read_csv("../data/market_master.csv", index_col="Date", parse_dates=True)

data = state_data.join(market_data[["nifty_ret"]], how="inner")
print(f"Dataset shape: {data.shape}")

# =============================================
# 2. REWARD FUNCTION
# =============================================
class RewardFunction:
    def __init__(self, cost=0.0005, risk_penalty=0.08, drawdown_penalty=0.25):
        self.cost = cost
        self.risk_penalty = risk_penalty
        self.drawdown_penalty = drawdown_penalty
    
    def compute(self, current_ret, prev_position, new_position, drawdown):
        strategy_ret = prev_position * current_ret
        cost = abs(new_position - prev_position) * self.cost
        
        risk_penalty = self.risk_penalty * (strategy_ret ** 2)
        dd_penalty = self.drawdown_penalty * abs(drawdown) if drawdown < -0.05 else 0
        
        reward = strategy_ret - cost - risk_penalty - dd_penalty
        net_ret = strategy_ret - cost
        
        return reward, net_ret

# =============================================
# 3. ENVIRONMENT
# =============================================
class QuantumAlphaEnv(gym.Env):
    def __init__(self, data, initial_balance=1_000_000):
        super().__init__()
        
        self.data = data.reset_index(drop=True)
        self.max_steps = len(self.data) - 1
        self.initial_balance = initial_balance
        
        self.feature_cols = [
            "mom_20_norm", "vol_signal_norm", "trend_signal_norm",
            "dd_signal_norm", "vix_signal_norm", "breadth_signal_norm"
        ]
        
        self.action_space = spaces.Discrete(3)
        
        self.observation_space = spaces.Box(
            low=-5, high=5,
            shape=(len(self.feature_cols) + 1,),
            dtype=np.float32
        )
        
        self.reward_fn = RewardFunction()
        self.reset()
    
    def reset(self):
        self.current_step = 0
        self.position = 0
        self.balance = self.initial_balance
        self.peak_balance = self.initial_balance
        return self._get_observation()
    
    def _get_observation(self):
        row = self.data.iloc[self.current_step]
        obs = row[self.feature_cols].values.astype(np.float32)
        
       
        return np.append(obs, self.position)
    
    def step(self, action):
        new_position = {0: 0, 1: 1, 2: -1}[action]
        prev_position = self.position
        
        current_ret = self.data.iloc[self.current_step]["nifty_ret"]
        current_drawdown = (self.balance / self.peak_balance) - 1
        
        reward, net_ret = self.reward_fn.compute(
            current_ret, prev_position, new_position, current_drawdown
        )
        
        self.balance *= (1 + net_ret)
        self.peak_balance = max(self.peak_balance, self.balance)
        
        self.position = new_position
        self.current_step += 1
        done = self.current_step >= self.max_steps
        
        return self._get_observation(), reward, done, {
            "balance": self.balance,
            "position": self.position,
            "drawdown": (self.balance / self.peak_balance) - 1
        }

# =============================================
# 4. CREATE ENV
# =============================================
env = QuantumAlphaEnv(data)

print("Environment created successfully")
print(f"Observation shape: {env.observation_space.shape}")
print(f"Action space: {env.action_space}")

# =============================================
# 5. DEVICE SETUP
# =============================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# =============================================
# 6. DQN MODEL
# =============================================
class DQN(nn.Module):
    def __init__(self, state_size, action_size, hidden=128):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, action_size)
        )
    
    def forward(self, x):
        return self.net(x)

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

policy_net = DQN(state_size, action_size).to(device)
target_net = DQN(state_size, action_size).to(device)

target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

print("DQN networks initialized")

# =============================================
# 7. SANITY CHECK
# =============================================
state = env.reset()
state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)

with torch.no_grad():
    q_values = policy_net(state_tensor)

print("\nSample Q-values:", q_values.cpu().numpy())

Dataset shape: (1529, 7)
Environment created successfully
Observation shape: (7,)
Action space: Discrete(3)
Using device: cpu
DQN networks initialized

Sample Q-values: [[-0.17445663 -0.10711184  0.00036846]]
